# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ayaahmed571/Flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window


**Unit of analysis:**
One row represents one content page (`content_hash_id`) for one client (`client_hash_id`) on one report date.

**Time window:**
This notebook uses data from **March 2026 (month = 2026-03)**, which is a mid-panel month. I chose a mid-panel month to avoid developing on the final month of the dataset.*

## 2. Fields: feature / label / context / excluded

### Features
- gsc_impressions
- gsc_clicks
- gsc_avg_position
- sessions_organic
- scroll_events

These features are available before making the content refresh decision.

### Label / Proxy
The goal is to rank or score pages that are likely to need a content refresh.

### Context fields
- report_date
- client_hash_id
- content_hash_id

These identify the page, client, and date but are not used as predictive features.

### Excluded fields
- client_hash_id
- content_hash_id
- Any future or label-derived columns

These are excluded because they either identify records or could leak future information.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
rel = "hf://datasets/FlyRank/internship-warehouse"

con.sql(f"""
SELECT COUNT(*) AS duplicate_rows
FROM (
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS cnt
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
    )
    GROUP BY
        report_date,
        client_hash_id,
        content_hash_id
    HAVING COUNT(*) > 1
)
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────┐
│ duplicate_rows │
│     int64      │
├────────────────┤
│              0 │
└────────────────┘



The result returned 0 duplicate rows, confirming that one row represents one page for one client on one date.

In [ ]:
con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM read_parquet(
    '{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
)
""").show()

┌────────────┬────────────┬────────────┐
│ total_rows │ start_date │  end_date  │
│   int64    │    date    │    date    │
├────────────┼────────────┼────────────┤
│    9841378 │ 2026-03-01 │ 2026-03-31 │
└────────────┴────────────┴────────────┘



This confirms the size of the selected data and that it covers March 2026.

In [ ]:
con.sql(f"""
SELECT
    COUNT(*) AS available_rows
FROM read_parquet(
    '{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
)
WHERE
    gsc_data_available IS TRUE
    AND ga4_data_available IS TRUE
""").show()

┌────────────────┐
│ available_rows │
│     int64      │
├────────────────┤
│         364347 │
└────────────────┘



Only rows with both GSC and GA4 available are used.

## 4. Data limits

- This notebook uses only one month (March 2026).
- Results may not generalize to all months.
- Pages without GSC or GA4 data are excluded.
- The notebook is intended for analysis and feature development, not final model evaluation.

## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.